In [2]:
import pandas as pd

data = {
    "machine_id": ["M001", "M002", "M003", "M004", "M005",
                   "M006", "M007", "M008", "M009", "M010"],

    "temperature_c": [62, 74, 88, 91, 69, 82, 77, 95, 58, 85],

    "vibration_mm_s": [1.8, 3.2, 6.5, 7.2, 2.1, 5.8, 4.1, 8.0, 1.2, 6.1],

    "load_pct": [42, 65, 88, 94, 38, 81, 72, 96, 35, 86],

    "pressure_bar": [5.1, 5.5, 6.1, 6.3, 5.0, 5.9, 5.7, 6.5, 4.9, 6.0],

    "hours_since_maintenance": [120, 350, 780, 920, 80,
                                640, 410, 1100, 50, 720]
}

df = pd.DataFrame(data)

print(df)

  machine_id  temperature_c  vibration_mm_s  load_pct  pressure_bar  \
0       M001             62             1.8        42           5.1   
1       M002             74             3.2        65           5.5   
2       M003             88             6.5        88           6.1   
3       M004             91             7.2        94           6.3   
4       M005             69             2.1        38           5.0   
5       M006             82             5.8        81           5.9   
6       M007             77             4.1        72           5.7   
7       M008             95             8.0        96           6.5   
8       M009             58             1.2        35           4.9   
9       M010             85             6.1        86           6.0   

   hours_since_maintenance  
0                      120  
1                      350  
2                      780  
3                      920  
4                       80  
5                      640  
6              

In [3]:
def high_temperature(x):
    if x <= 70:
        return 0

    if x >= 90:
        return 1

    return (x - 70) / (90 - 70)

In [4]:
print(high_temperature(60))
print(high_temperature(70))
print(high_temperature(80))
print(high_temperature(90))
print(high_temperature(95))

0
0
0.5
1
1


In [5]:
def high_vibration(x):
    if x <= 3:
        return 0

    if x >= 7:
        return 1

    return (x - 3) / (7 - 3)

In [6]:
df["temp_high"] = df["temperature_c"].apply(high_temperature)

df["vibration_high"] = df["vibration_mm_s"].apply(high_vibration)

print(df[[
    "machine_id",
    "temperature_c",
    "temp_high",
    "vibration_mm_s",
    "vibration_high"
]])

  machine_id  temperature_c  temp_high  vibration_mm_s  vibration_high
0       M001             62       0.00             1.8           0.000
1       M002             74       0.20             3.2           0.050
2       M003             88       0.90             6.5           0.875
3       M004             91       1.00             7.2           1.000
4       M005             69       0.00             2.1           0.000
5       M006             82       0.60             5.8           0.700
6       M007             77       0.35             4.1           0.275
7       M008             95       1.00             8.0           1.000
8       M009             58       0.00             1.2           0.000
9       M010             85       0.75             6.1           0.775


In [7]:
def high_load(x):
    if x <= 60:
        return 0

    if x >= 90:
        return 1

    return (x - 60) / (90 - 60)

In [8]:
df["load_high"] = df["load_pct"].apply(high_load)

In [9]:
df["rule_1"] = df[["temp_high", "vibration_high"]].min(axis=1)
df["rule_2"] = df[["vibration_high", "load_high"]].min(axis=1)

In [10]:
def old_maintenance(x):
    if x <= 300:
        return 0

    if x >= 900:
        return 1

    return (x - 300) / (900 - 300)

In [11]:
df["maintenance_old"] = (
    df["hours_since_maintenance"]
    .apply(old_maintenance)
)

In [12]:
df["rule_3"] = df[
    ["temp_high", "maintenance_old"]
].min(axis=1)

In [13]:
df["risk_membership"] = df[
    ["rule_1", "rule_2", "rule_3"]
].max(axis=1)

In [14]:
df["risk_score"] = df["risk_membership"] * 100

In [18]:
from IPython.display import display

def style_risk_table(table, title):
    number_formats = {
        "temperature_c": "{:.1f}",
        "vibration_mm_s": "{:.1f}",
        "load_pct": "{:.0f}%",
        "hours_since_maintenance": "{:.0f}",
        "risk_score": "{:.1f}%",
    }
    return (
        table.style
        .format(number_formats)
        .set_properties(**{
            "text-align": "center",
            "padding": "6px 10px",
            "border": "1px solid #d9e2ec",
        })
        .set_properties(
            subset=["risk_score"],
            **{"background-color": "#fff3cd", "font-weight": "bold"},
        )
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "16px"),
                    ("font-weight", "bold"),
                    ("color", "#243b53"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#243b53"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                ],
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#f5f7fa")],
            },
        ])
        .set_caption(title)
        .hide(axis="index")
    )

display(style_risk_table(result, "Machine Temperature and Risk Summary"))

machine_id,temperature_c,vibration_mm_s,load_pct,hours_since_maintenance,temp_high,vibration_high,load_high,maintenance_old,risk_score
M001,62.0,1.8,42%,120,0.000000,0.000000,0.000000,0.000000,0.0%
M002,74.0,3.2,65%,350,0.200000,0.050000,0.166667,0.083333,8.3%
M003,88.0,6.5,88%,780,0.900000,0.875000,0.933333,0.800000,87.5%
M004,91.0,7.2,94%,920,1.000000,1.000000,1.000000,1.000000,100.0%
M005,69.0,2.1,38%,80,0.000000,0.000000,0.000000,0.000000,0.0%
M006,82.0,5.8,81%,640,0.600000,0.700000,0.700000,0.566667,70.0%
M007,77.0,4.1,72%,410,0.350000,0.275000,0.400000,0.183333,27.5%
M008,95.0,8.0,96%,1100,1.000000,1.000000,1.000000,1.000000,100.0%
M009,58.0,1.2,35%,50,0.000000,0.000000,0.000000,0.000000,0.0%
M010,85.0,6.1,86%,720,0.750000,0.775000,0.866667,0.700000,77.5%


In [19]:
high_risk = df[df["risk_score"] >= 70][[
    "machine_id",
    "temperature_c",
    "vibration_mm_s",
    "load_pct",
    "risk_score"
]]

display(style_risk_table(high_risk, "High-Risk Machines (Risk Score >= 70%)"))

machine_id,temperature_c,vibration_mm_s,load_pct,risk_score
M003,88.0,6.5,88%,87.5%
M004,91.0,7.2,94%,100.0%
M006,82.0,5.8,81%,70.0%
M008,95.0,8.0,96%,100.0%
M010,85.0,6.1,86%,77.5%
